# 제주 특산물 가격 예측 - 1등 코드 기반 재설계 v4.2.0

| 항목 | 내용 |
|------|------|
| **버전** | v4.2.0 |
| **날짜** | 2026-03-16 |
| **모델** | CatBoost + LightGBM (VotingRegressor 방식) |
| **검증** | **없음 — 전체 데이터 학습** |
| **전처리** | 1등 코드 기반 (최소 피처, TG만 sqrt, non-TG 변환 없음) |
| **출력** | results/submission_v4.2.0.csv |

## v4.2.0 변경 내용 (1등 코드 기반 재설계)

| 항목 | v4.1.0 | v4.2.0 | 근거 |
|------|--------|--------|------|
| 타겟 변환 | TG: sqrt, non-TG: **log1p** | TG: sqrt, non-TG: **변환 없음** | 1등 코드 동일 방식 |
| 학습 데이터 | 2022-03 제외 (holdout) | **전체 59,397행** | 1등 코드 동일 방식 |
| 피처 | 시간+sin/cos+EMA | **시간 기본 7개만** | 1등 코드 동일 방식 |
| EMA 피처 | 7/30/60/120일 | **제거** | v4.1.0 실패 원인 |
| sin/cos 인코딩 | 있음 | **제거** | 1등 코드에 없음 |
| quarter/is_weekend | 있음 | **제거** | 1등 코드에 없음 |
| depth | 7 | **10** | 1등 코드 동일 |
| learning_rate | 0.03 | **0.01** | 1등 코드 동일 |
| n_estimators | 5000+early_stop | **1000 고정** | 검증셋 없으므로 early_stop 불가 |
| TG 처리 | 단일 모델 | **2개 모델 앙상블** | 1등 코드 동일 전략 |

## 프레이밍 차이 기록

| 관점 | 1등 (이번 v4.2.0) | 기존 (v4.0~4.1) |
|------|------------------|----------------|
| 문제 유형 | **표 데이터 회귀** | **시계열 예측** |
| 시간 순서 | 고려 안 함 | 엄격히 유지 |
| 검증 | 없음 | 2022-03 holdout |
| 래그 피처 | 없음 | EMA 추세 반영 |
| 지배적 신호 | 달력 패턴(seasonality) | 가격 추세(trend) |

> v1.0.1 Public 658.6 결과가 이 데이터의 지배적 신호가 달력 패턴임을 증명함

---
## 버전 히스토리

| 버전 | 모델 | 핵심 변경 | Public | Private |
|------|------|----------|---------|---------|
| v1.0.1 | DNN | DACON 1위 전처리 | **658.6** ✅ | 825.0 |
| v1.2.0 | DNN+Emb+Res | Residual Block | 1155.3 ❌ | 1326.3 |
| v3.1.0 | DNN+CB+XGB | 앙상블 | 910.9 ❌ | 1062.6 |
| v4.1.0 | LGB+CB | EMA 피처 추가 | 1170.5 ❌ | 1241.9 |
| **v4.2.0** | **LGB+CB** | **1등 기반 재설계** | — | — |

---
## 1. 라이브러리 로드

In [ ]:
try:
    import lightgbm as lgb
except ImportError:
    import subprocess; subprocess.run(['pip', 'install', 'lightgbm'], check=True)
    import lightgbm as lgb

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, datetime, random, os, platform
warnings.filterwarnings('ignore')

import holidays
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 2024
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

N_JOBS = max(1, int(os.cpu_count() * 0.8))
print(f'사용 스레드: {N_JOBS}')
print(f'LightGBM: {lgb.__version__} | CatBoost: {__import__("catboost").__version__}')

---
## 2. 데이터 로드

In [ ]:
DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')
print(f'train: {train.shape}, test: {test.shape}')

---
## 3. 전처리 (1등 코드 기반)

### 1등 코드와 동일한 피처만 사용
```
year, month, day, week_day, year_month, week_num, holiday
+ item, corporation, location (one-hot)
```
제거된 피처: quarter, is_weekend, day_of_year, sin/cos 인코딩, EMA

In [ ]:
def pre_all(train, test):
    """1등 코드와 동일한 전처리"""
    train['timestamp'] = pd.to_datetime(train['timestamp'])
    test['timestamp']  = pd.to_datetime(test['timestamp'])
    df = pd.concat([train, test]).reset_index(drop=True)
    df.rename(columns={'supply(kg)': 'supply', 'price(원/kg)': 'price'}, inplace=True)

    df['year']     = df['timestamp'].dt.year
    df['month']    = df['timestamp'].dt.month
    df['day']      = df['timestamp'].dt.day
    df['week_day'] = df['timestamp'].dt.weekday

    le = LabelEncoder()
    df['year_month'] = df['timestamp'].map(lambda x: f'{x.year}-{x.month}')
    df['year_month'] = le.fit_transform(df['year_month'])

    df['week'] = df['timestamp'].map(
        lambda x: datetime.datetime(x.year, x.month, x.day).isocalendar()[1]
    )
    week_offsets = {2019: 0, 2020: 52, 2021: 52+53, 2022: 52+53+53, 2023: 52+53+53+52}
    df['week_num'] = df.apply(lambda r: int(r['week']) + week_offsets.get(r['year'], 0), axis=1)
    df.loc[df['timestamp'] == '2019-12-30', 'week_num'] = 52
    df.loc[df['timestamp'] == '2019-12-31', 'week_num'] = 52

    kr_holi = holidays.KR()
    df['holiday'] = df['timestamp'].map(lambda x: 1 if x in kr_holi else 0)

    train_out = df[~df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    test_out  = df[ df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    print(f'전처리 완료 — train: {train_out.shape}, test: {test_out.shape}')
    return train_out, test_out

train_pre, test_pre = pre_all(train, test)

# ── 이상치 처리 (1등 코드 동일) ──
outlier_thresholds = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}
for item, thr in outlier_thresholds.items():
    idx = train_pre[(train_pre['item'] == item) & (train_pre['price'] > thr)].index
    if len(idx):
        mean_p = train_pre[(train_pre['item'] == item) & (train_pre['price'] != 0)]['price'].mean()
        train_pre.loc[idx, 'price'] = mean_p
        print(f'{item}: {len(idx)}개 이상치 → 평균({mean_p:.0f}원)')

# ── TG 공휴일 보정 (1등 코드 동일) ──
tg_mask     = (train_pre['item'] == 'TG') & (train_pre['holiday'] == 1) & (train_pre['price'] != 0)
active_holi = list(train_pre[tg_mask].groupby('timestamp').count().reset_index()['timestamp'])
fix_idx     = train_pre[train_pre['timestamp'].isin(active_holi)].index
train_pre.loc[fix_idx, 'holiday'] = 0
print(f'TG 공휴일 보정: {len(fix_idx)}건')

print('\n[피처 목록]')
print('시간: year, month, day, week_day, year_month, week_num, holiday')
print('범주: item, corporation, location (one-hot 인코딩)')
print('제거: quarter, is_weekend, day_of_year, sin/cos, EMA')

---
## 4. 피처 준비 (one-hot 인코딩)

1등 코드는 CatBoost에 cat_features를 쓰지 않고 get_dummies로 one-hot 인코딩합니다.  
LightGBM도 동일하게 one-hot 방식으로 통일합니다.

In [ ]:
BASE_FEATS = ['year', 'month', 'day', 'week_day', 'year_month', 'week_num', 'holiday']

def make_dummies(df, drop_cols=None):
    """item, corporation, location one-hot 인코딩"""
    out = pd.get_dummies(df, columns=['item', 'corporation', 'location'])
    return out

# 전체 train + test 합쳐서 동일한 dummy 컬럼 보장
train_pre['_is_train'] = 1
test_pre['_is_train']  = 0
combined = pd.concat([train_pre, test_pre], axis=0).reset_index(drop=True)
combined = make_dummies(combined)

train_dum = combined[combined['_is_train'] == 1].drop(columns=['_is_train']).reset_index(drop=True)
test_dum  = combined[combined['_is_train'] == 0].drop(columns=['_is_train']).reset_index(drop=True)

# one-hot 이후 피처 컬럼 목록
DROP_COLS  = ['ID', 'timestamp', 'price', 'supply', 'week']
FEAT_COLS  = [c for c in train_dum.columns if c not in DROP_COLS]

print(f'전체 피처 수: {len(FEAT_COLS)}개')
print(FEAT_COLS)

---
## 5. TG / non-TG 분리

### TG 전략 (1등 코드 기반)
- **모델 A**: item one-hot 포함 (item_TG=1) — 전체 데이터에서 TG 행만 학습
- **모델 B**: item 컬럼 제외 — TG만의 패턴을 item 정보 없이 학습
- **TG 최종 예측** = (A + B) / 2

### non-TG 전략
- CR, CB, RD, BC 4개 품목 단일 모델
- **타겟 변환 없음** (1등 코드: non-TG는 원본 price 그대로)

In [ ]:
# ── TG 분리 ──
tg_mask_tr  = train_dum['item_TG'] == 1
tg_mask_te  = test_dum['item_TG']  == 1

# TG 학습/테스트
X_tg_tr = train_dum[tg_mask_tr][FEAT_COLS].copy()
y_tg_tr = train_dum[tg_mask_tr]['price'].copy()
y_tg_sqrt = np.sqrt(y_tg_tr)               # TG만 sqrt 변환
X_tg_te = test_dum[tg_mask_te][FEAT_COLS].copy()

# TG 모델 B용: item 컬럼 제거
ITEM_COLS = [c for c in FEAT_COLS if c.startswith('item_')]
FEAT_NO_ITEM = [c for c in FEAT_COLS if not c.startswith('item_')]
X_tg_tr_noitem = X_tg_tr[FEAT_NO_ITEM].copy()
X_tg_te_noitem = X_tg_te[FEAT_NO_ITEM].copy()

# ── non-TG 분리 ──
nontg_mask_tr = train_dum['item_TG'] == 0
nontg_mask_te = test_dum['item_TG']  == 0

X_nontg_tr = train_dum[nontg_mask_tr][FEAT_COLS].copy()
y_nontg_tr = train_dum[nontg_mask_tr]['price'].copy()  # 변환 없음
X_nontg_te = test_dum[nontg_mask_te][FEAT_COLS].copy()

print(f'TG    학습: {len(X_tg_tr):,}행 | 테스트: {len(X_tg_te):,}행')
print(f'nonTG 학습: {len(X_nontg_tr):,}행 | 테스트: {len(X_nontg_te):,}행')
print(f'\nTG 타겟: sqrt 변환 | nonTG 타겟: 원본 price')

---
## 6. 모델 파라미터 (1등 코드 기반)

| 파라미터 | 1등 CatBoost | v4.2.0 CatBoost | v4.2.0 LightGBM |
|----------|-------------|-----------------|------------------|
| n_estimators | 1000 | 1000 | 1000 |
| learning_rate | 0.01 | 0.01 | 0.01 |
| depth / num_leaves | 10 | 10 | 255 (≈ depth 10) |
| early_stopping | 없음 | 없음 | 없음 |

> 전체 데이터로 학습하므로 early_stopping용 검증셋 없음 → n_estimators=1000 고정

In [ ]:
def make_cb(seed=SEED):
    return CatBoostRegressor(
        iterations=1000,
        learning_rate=0.01,
        depth=10,
        l2_leaf_reg=3,
        loss_function='MAE',
        random_seed=seed,
        thread_count=N_JOBS,
        verbose=200
    )

def make_lgb(seed=SEED):
    return lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        num_leaves=255,
        max_depth=10,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2.0,
        objective='regression_l1',
        random_state=seed,
        n_jobs=N_JOBS,
        verbose=-1
    )

print('모델 파라미터 설정 완료')

---
## 7. TG 모델 학습 (모델 A + 모델 B)

### 모델 A: item one-hot 포함
### 모델 B: item 컬럼 제외

In [ ]:
print('=== TG 모델 A (item 포함) ===')
print('  CatBoost 학습...')
cb_tg_A = make_cb()
cb_tg_A.fit(X_tg_tr, y_tg_sqrt)

print('  LightGBM 학습...')
lgb_tg_A = make_lgb()
lgb_tg_A.fit(X_tg_tr, y_tg_sqrt)

pred_tg_A = (cb_tg_A.predict(X_tg_te) + lgb_tg_A.predict(X_tg_te)) / 2
pred_tg_A = np.power(np.clip(pred_tg_A, 0, None), 2)  # sqrt 역변환
print(f'  모델 A 예측 완료 | 평균: {pred_tg_A.mean():.1f}원')

In [ ]:
print('=== TG 모델 B (item 제외) ===')
print('  CatBoost 학습...')
cb_tg_B = make_cb()
cb_tg_B.fit(X_tg_tr_noitem, y_tg_sqrt)

print('  LightGBM 학습...')
lgb_tg_B = make_lgb()
lgb_tg_B.fit(X_tg_tr_noitem, y_tg_sqrt)

pred_tg_B = (cb_tg_B.predict(X_tg_te_noitem) + lgb_tg_B.predict(X_tg_te_noitem)) / 2
pred_tg_B = np.power(np.clip(pred_tg_B, 0, None), 2)
print(f'  모델 B 예측 완료 | 평균: {pred_tg_B.mean():.1f}원')

# TG 최종: 모델 A + 모델 B 평균
pred_tg_final = (pred_tg_A + pred_tg_B) / 2
print(f'\nTG 최종 예측 (A+B)/2 | 평균: {pred_tg_final.mean():.1f}원')

---
## 8. non-TG 모델 학습

CR, CB, RD, BC — **타겟 변환 없음, 원본 price 그대로**

In [ ]:
print('=== non-TG 모델 (CR / CB / RD / BC) ===')
print('  타겟: 원본 price (변환 없음)')

print('  CatBoost 학습...')
cb_nontg = make_cb()
cb_nontg.fit(X_nontg_tr, y_nontg_tr)

print('  LightGBM 학습...')
lgb_nontg = make_lgb()
lgb_nontg.fit(X_nontg_tr, y_nontg_tr)

pred_nontg = (cb_nontg.predict(X_nontg_te) + lgb_nontg.predict(X_nontg_te)) / 2
pred_nontg = np.clip(pred_nontg, 0, None)
print(f'  non-TG 예측 완료 | 평균: {pred_nontg.mean():.1f}원')

---
## 9. 피처 중요도

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# non-TG CatBoost
cb_imp = pd.Series(cb_nontg.get_feature_importance(), index=FEAT_COLS).sort_values(ascending=False)
cb_imp.head(12).plot(kind='barh', ax=axes[0], color='darkorange', alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_title('nonTG CatBoost Top-12', fontsize=12)
axes[0].grid(axis='x', alpha=0.3)

# non-TG LightGBM
lgb_imp = pd.Series(lgb_nontg.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)
lgb_imp.head(12).plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_title('nonTG LightGBM Top-12', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('v4.2.0 피처 중요도', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('./results', exist_ok=True)
plt.savefig('./results/feature_importance_v4.2.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. 후처리 + 제출 파일 생성

후처리 임계값은 1등 코드와 동일

In [ ]:
# TG 결과
tg_result = test_dum[tg_mask_te][['ID']].copy()
tg_result['answer']   = pred_tg_final
tg_result['item']     = 'TG'

# non-TG 결과
nontg_items = test_pre[test_pre['item'] != 'TG']['item'].values
nontg_result = test_dum[nontg_mask_te][['ID']].copy()
nontg_result['answer'] = pred_nontg
nontg_result['item']   = nontg_items

result_df = pd.concat([tg_result, nontg_result], axis=0).reset_index(drop=True)

# 후처리: 아이템별 임계값 미만 → 0 (1등 코드 동일)
min_thresholds = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for item, thr in min_thresholds.items():
    mask = (result_df['item'] == item) & (result_df['answer'] < thr)
    result_df.loc[mask, 'answer'] = 0.0
    if mask.sum() > 0:
        print(f'{item}: {mask.sum()}개 → 0 처리')

print('\n예측 통계:')
print(result_df.groupby('item')['answer'].agg(['mean', 'min', 'max']).round(1))

# 제출 파일
result = sub[['ID']].merge(result_df[['ID', 'answer']], on='ID', how='left')
result['answer'] = result['answer'].fillna(0.0)

SUBMISSION_PATH = './results/submission_v4.2.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')
print(f'\n저장 완료: {SUBMISSION_PATH} ({len(result)}행)')

---
## 11. 결과 요약

In [ ]:
print('=' * 65)
print('  v4.2.0 최종 결과')
print('=' * 65)
print(f'  [학습 전략]  전체 데이터 학습 (검증 분리 없음)')
print(f'  [TG 모델]    모델A(item 포함) + 모델B(item 제외) 평균')
print(f'  [nonTG 모델] 단일 모델, 타겟 변환 없음')
print(f'  [피처]       기본 7개 시간 피처 + one-hot (sin/cos·EMA 제거)')
print(f'  [depth]      10  |  [lr] 0.01  |  [iter] 1000')
print()
print('  ※ 검증셋 없이 전체 학습 → 내부 MAE 없음')
print('  ※ 성능 확인은 DACON 제출 후 Public Score로만 가능')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print('=' * 65)

---
### 다음 버전

| 버전 | 내용 |
|------|------|
| **v4.3.0** | 명절 접근 거리 피처 추가 (dist_to_seollal, dist_to_chuseok) — 박효준님 제안 |
| **v4.4.0** | 품목별 개별 모델 (TG / CR+RD / CB+BC) — 김대원님 제안 |
| **v5.0.0** | Optuna 하이퍼파라미터 최적화 |